# DeepSets vs Graph Neural Network Comparison

## Research-Backed Methodology

This notebook implements a comprehensive, research-backed comparison between DeepSets (permutation-invariant) and GraphSAGE (GNN) models trained on surface code error correction extrapolation.

**Key Principles:**
- Fair comparison with parameter budget matching
- Statistical significance testing (Wilcoxon signed-rank)
- Identical evaluation protocols
- Multiple performance indices (accuracy, speed, memory, complexity)

**Distances Analyzed:** d=3, d=5, d=7, d=9, d=11, d=13

## 1. Imports and Setup

In [ ]:
import sys
import json
import time
import gc
import os
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import sklearn
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy import stats
from scipy.stats import wilcoxon, chi2_contingency
import math
import stim

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/results/deepsets_vs_gnn_comparison -> code/

sys.path.insert(0, str(BASE_PATH))

# Import model classes
from benchmark_models import DeepSets, SurfaceCodeSampler
from models import GraphSAGE, SparseGraph

# Set up paths
RESULTS_DIR = BASE_PATH / "results" / "deepsets_vs_gnn_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_RESULTS_DIR = RESULTS_DIR / "results"
OUTPUT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  OUTPUT_RESULTS_DIR: {OUTPUT_RESULTS_DIR}")
print(f"  PLOTS_DIR: {PLOTS_DIR}")

## 2. Configuration

In [ ]:
# ============================================================
# DATA MODE CONFIGURATION
# ============================================================
USE_SAVED_DATA = False

# Final plots directory
FINAL_PLOTS_DIR = PLOTS_DIR / "final"
FINAL_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
DISTANCES = [3, 5, 7, 9, 11, 13]
EXPERIMENT_NAME = "equal_333333"  # Model trained on balanced d=3,5,7 data

# Test dataset configuration
TEST_SAMPLES_PER_DISTANCE = 10000
BENCHMARK_SAMPLES = 10000

# Inference benchmarking configuration
BATCH_SIZES = [1, 8, 16, 32, 64, 128, 256]
NUM_WARMUP_RUNS = 10
NUM_BENCHMARK_RUNS = 5
NUM_ACCURACY_RUNS = 4

# Statistical test significance level
ALPHA = 0.05

# Random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Configuration:")
print(f"  USE_SAVED_DATA: {USE_SAVED_DATA}")
print(f"  FINAL_PLOTS_DIR: {FINAL_PLOTS_DIR}")
print(f"  EXPERIMENT_NAME: {EXPERIMENT_NAME}")
print(f"  Distances: {DISTANCES}")
print(f"  Test samples per distance: {TEST_SAMPLES_PER_DISTANCE:,}")
print(f"  Benchmark samples: {BENCHMARK_SAMPLES:,}")
print(f"  Batch sizes for benchmarking: {BATCH_SIZES}")
print(f"  Number of benchmark runs: {NUM_BENCHMARK_RUNS}")
print(f"  Number of accuracy runs: {NUM_ACCURACY_RUNS}")
print(f"  Significance level (alpha): {ALPHA}")
print(f"  Random seed: {SEED}")

## 3. Helper Functions

In [ ]:
def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_model_size_mb(model_path: Path) -> float:
    """Get model file size in MB."""
    if model_path.exists():
        return model_path.stat().st_size / (1024 * 1024)
    return 0.0


def load_deepsets_model(experiment_name: str, base_path: Path) -> Tuple:
    """Load DeepSets extrapolation model."""
    model_path = base_path / "deepsets" / "extrapolation" / "models" / "revised_training" / f"{experiment_name}.pt"
    
    if not model_path.exists():
        raise FileNotFoundError(f"DeepSets model not found at {model_path}")
    
    # Load checkpoint to get config
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {})
    
    # Initialize model with correct architecture
    model = DeepSets(
        nickname=experiment_name,
        phi_hidden=config.get('phi_hidden', [64, 128, 256]),
        rho_hidden=config.get('rho_hidden', [256, 128, 64]),
        pool=config.get('pool', 'mean'),
        dropout=config.get('dropout', 0.1),
        device=device,
        base_path=base_path
    )
    
    # Load state dict
    model.model.load_state_dict(checkpoint['state_dict'])
    model.model.eval()
    model._config = config
    
    num_params = count_parameters(model.model)
    model_size_mb = get_model_size_mb(model_path)
    
    info = {
        'experiment': experiment_name,
        'num_parameters': num_params,
        'model_size_mb': model_size_mb,
        'model_path': str(model_path),
        'config': config
    }
    
    return model, info


def load_graphsage_model(experiment_name: str, base_path: Path) -> Tuple:
    """Load GraphSAGE extrapolation model."""
    model_path = base_path / "gSAGE" / "extrapolation" / "models" / "revised_training" / f"{experiment_name}.pt"
    
    if not model_path.exists():
        raise FileNotFoundError(f"GraphSAGE model not found at {model_path}")
    
    # Load checkpoint to get config
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {
        'in_channels': 5,
        'hidden_dim': 128,
        'num_layers': 5,
        'dropout': 0.0,
        'aggr': 'max'
    })
    
    # Initialize model
    model = GraphSAGE(
        nickname=experiment_name,
        in_channels=config.get('in_channels', 5),
        hidden_dim=config.get('hidden_dim', 128),
        num_layers=config.get('num_layers', 5),
        dropout=config.get('dropout', 0.0),
        aggr=config.get('aggr', 'max'),
        device=device,
        base_path=base_path
    )
    
    # Load state dict
    if 'state_dict' in checkpoint:
        model.model.load_state_dict(checkpoint['state_dict'])
    model.model.eval()
    
    num_params = count_parameters(model.model)
    model_size_mb = get_model_size_mb(model_path)
    
    info = {
        'experiment': experiment_name,
        'num_parameters': num_params,
        'model_size_mb': model_size_mb,
        'model_path': str(model_path),
        'config': config
    }
    
    return model, info


print("Helper functions defined.")

## 4. Load Models

In [ ]:
# Load DeepSets Model
print(f"Loading DeepSets model ({EXPERIMENT_NAME})...")
try:
    deepsets_model, deepsets_info = load_deepsets_model(EXPERIMENT_NAME, BASE_PATH)
    print(f"  ✓ DeepSets: {deepsets_info['num_parameters']:,} params, {deepsets_info['model_size_mb']:.2f} MB")
except Exception as e:
    print(f"  ✗ DeepSets: Failed to load - {e}")
    deepsets_model = None
    deepsets_info = {}

# Load GraphSAGE Model
print(f"\nLoading GraphSAGE model ({EXPERIMENT_NAME})...")
try:
    graphsage_model, graphsage_info = load_graphsage_model(EXPERIMENT_NAME, BASE_PATH)
    print(f"  ✓ GraphSAGE: {graphsage_info['num_parameters']:,} params, {graphsage_info['model_size_mb']:.2f} MB")
except Exception as e:
    print(f"  ✗ GraphSAGE: Failed to load - {e}")
    graphsage_model = None
    graphsage_info = {}

print(f"\nLoaded models successfully")

## 5. Parameter Budget Analysis

In [ ]:
# Create parameter comparison table
param_comparison = []
for d in DISTANCES:
    param_comparison.append({
        'distance': d,
        'deepsets_params': deepsets_info.get('num_parameters', 0),
        'graphsage_params': graphsage_info.get('num_parameters', 0),
        'deepsets_size_mb': deepsets_info.get('model_size_mb', 0),
        'graphsage_size_mb': graphsage_info.get('model_size_mb', 0),
        'param_ratio': graphsage_info.get('num_parameters', 0) / deepsets_info.get('num_parameters', 1) if deepsets_info.get('num_parameters', 0) > 0 else 0
    })

param_df = pd.DataFrame(param_comparison)
print("Parameter Budget Comparison:")
print(param_df.to_string(index=False))

if len(param_comparison) > 0:
    avg_ratio = param_df['param_ratio'].mean()
    print(f"\nAverage parameter ratio (GraphSAGE/DeepSets): {avg_ratio:.2f}x")
    print(f"DeepSets params: {deepsets_info.get('num_parameters', 0):,}")
    print(f"GraphSAGE params: {graphsage_info.get('num_parameters', 0):,}")

## 6. Generate Shared Test Sets

In [ ]:
# Generate shared test datasets for fair comparison
shared_test_data = {}
graph_builder = SparseGraph(k_neighbors=6, device=torch.device('cpu'))
sampler = SurfaceCodeSampler(p=0.005, device=torch.device('cpu'))

print("Generating shared test datasets...")
for d in DISTANCES:
    # Generate detections with fixed seed
    torch.manual_seed(SEED + d)
    np.random.seed(SEED + d)
    
    detections, labels = sampler.sample(
        d=d,
        num_samples=TEST_SAMPLES_PER_DISTANCE,
        p_values=[0.001, 0.003, 0.005, 0.007],
        p_weights=[0.25, 0.25, 0.25, 0.25]
    )
    
    # Convert to graphs for GraphSAGE
    graphs = graph_builder.batch_to_pyg(detections, labels)
    
    shared_test_data[d] = {
        'detections': detections.cpu(),
        'labels': labels.cpu(),
        'graphs': graphs,
        'num_samples': len(labels)
    }
    
    print(f"  ✓ d={d}: {len(labels):,} samples")

print(f"\nGenerated shared test data for {len(shared_test_data)} distances")

## 7. Inference Speed Benchmarking

In [ ]:
def benchmark_inference_speed(
    model,
    data,
    batch_sizes: List[int],
    num_warmup: int = 10,
    num_runs: int = 5,
    is_graph_model: bool = False,
    is_deepsets: bool = False,
    distance: int = None
) -> Dict:
    """Benchmark inference speed with multiple runs."""
    model.model.eval()
    model.model.to(device)
    
    results = {}
    
    for batch_size in batch_sizes:
        if is_graph_model:
            loader = DataLoader(data, batch_size=batch_size, shuffle=False)
            num_samples = len(data)
        else:
            num_samples = len(data)
        
        # Warmup
        with torch.no_grad():
            warmup_count = 0
            if is_graph_model:
                for batch in loader:
                    if warmup_count >= num_warmup:
                        break
                    batch = batch.to(device)
                    _ = model.model(batch)
                    warmup_count += 1
            elif is_deepsets:
                for i in range(0, min(num_warmup * batch_size, num_samples), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    _ = model.predict(batch, distance)
            else:
                for i in range(0, min(num_warmup * batch_size, num_samples), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    _ = model.model(batch)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        # Timed runs
        run_times = []
        for run in range(num_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            start_time = time.time()
            
            with torch.no_grad():
                if is_graph_model:
                    for batch in loader:
                        batch = batch.to(device)
                        _ = model.model(batch)
                elif is_deepsets:
                    for i in range(0, num_samples, batch_size):
                        batch = data[i:i+batch_size].to(device)
                        _ = model.predict(batch, distance)
                else:
                    for i in range(0, num_samples, batch_size):
                        batch = data[i:i+batch_size].to(device)
                        _ = model.model(batch)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            run_times.append(time.time() - start_time)
        
        mean_time = np.mean(run_times)
        std_time = np.std(run_times)
        throughput = num_samples / mean_time
        latency_per_sample = (mean_time / num_samples) * 1e6
        
        results[batch_size] = {
            'mean_time_seconds': mean_time,
            'std_time_seconds': std_time,
            'throughput_samples_per_sec': throughput,
            'latency_per_sample_us': latency_per_sample,
            'runs': run_times
        }
    
    return results

# Benchmark models
print("Benchmarking inference speed...")
inference_benchmarks = {}

for d in DISTANCES:
    inference_benchmarks[d] = {}
    
    # Benchmark DeepSets
    if deepsets_model:
        print(f"  Benchmarking DeepSets d={d}...")
        detections = shared_test_data[d]['detections']
        results = benchmark_inference_speed(
            deepsets_model,
            detections,
            BATCH_SIZES,
            NUM_WARMUP_RUNS,
            NUM_BENCHMARK_RUNS,
            is_graph_model=False,
            is_deepsets=True,
            distance=d
        )
        inference_benchmarks[d]['deepsets'] = results
    
    # Benchmark GraphSAGE
    if graphsage_model:
        print(f"  Benchmarking GraphSAGE d={d}...")
        graphs = shared_test_data[d]['graphs']
        results = benchmark_inference_speed(
            graphsage_model,
            graphs,
            BATCH_SIZES,
            NUM_WARMUP_RUNS,
            NUM_BENCHMARK_RUNS,
            is_graph_model=True
        )
        inference_benchmarks[d]['graphsage'] = results

print("\n✓ Inference benchmarking complete")

## 8. Accuracy Evaluation

In [ ]:
def evaluate_accuracy_metrics(
    model,
    data,
    labels,
    num_runs: int = 4,
    threshold: float = 0.5,
    is_graph_model: bool = False,
    is_deepsets: bool = False,
    distance: int = None,
    batch_size: int = 256
) -> Dict:
    """Evaluate accuracy metrics with multiple runs."""
    model.model.eval()
    model.model.to(device)
    
    all_accuracies = []
    all_precisions = []
    all_recalls = []
    all_f1s = []
    all_predictions = []
    
    for run in range(num_runs):
        with torch.no_grad():
            predictions = []
            
            if is_graph_model:
                loader = DataLoader(data, batch_size=batch_size, shuffle=False)
                for batch in loader:
                    batch = batch.to(device)
                    pred = model.model(batch)
                    predictions.append(pred.cpu())
            elif is_deepsets:
                for i in range(0, len(data), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    pred = model.predict(batch, distance)
                    predictions.append(pred.cpu())
            else:
                for i in range(0, len(data), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    pred = model.model(batch)
                    predictions.append(pred.cpu())
            
            predictions = torch.cat(predictions, dim=0).squeeze()
            binary_preds = (predictions >= threshold).float()
            labels_tensor = labels.float()
            
            correct = (binary_preds == labels_tensor).sum().item()
            accuracy = correct / len(labels_tensor)
            
            tp = ((binary_preds == 1) & (labels_tensor == 1)).sum().item()
            fp = ((binary_preds == 1) & (labels_tensor == 0)).sum().item()
            fn = ((binary_preds == 0) & (labels_tensor == 1)).sum().item()
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
            
            all_accuracies.append(accuracy)
            all_precisions.append(precision)
            all_recalls.append(recall)
            all_f1s.append(f1)
            
            if run == 0:
                all_predictions = binary_preds.numpy()
    
    mean_acc = np.mean(all_accuracies)
    std_acc = np.std(all_accuracies)
    ci_acc = 1.96 * std_acc / np.sqrt(num_runs)
    
    return {
        'accuracy_mean': mean_acc,
        'accuracy_std': std_acc,
        'accuracy_ci_95': ci_acc,
        'precision_mean': np.mean(all_precisions),
        'precision_std': np.std(all_precisions),
        'recall_mean': np.mean(all_recalls),
        'recall_std': np.std(all_recalls),
        'f1_mean': np.mean(all_f1s),
        'f1_std': np.std(all_f1s),
        'f1_ci_95': 1.96 * np.std(all_f1s) / np.sqrt(num_runs),
        'all_accuracies': all_accuracies,
        'all_f1s': all_f1s,
        'predictions': all_predictions
    }

# Evaluate models
print("Evaluating accuracy metrics...")
accuracy_results = {}

for d in DISTANCES:
    accuracy_results[d] = {}
    labels = shared_test_data[d]['labels']
    
    # Evaluate DeepSets
    if deepsets_model:
        print(f"  Evaluating DeepSets d={d}...")
        detections = shared_test_data[d]['detections']
        results = evaluate_accuracy_metrics(
            deepsets_model,
            detections,
            labels,
            NUM_ACCURACY_RUNS,
            is_graph_model=False,
            is_deepsets=True,
            distance=d
        )
        accuracy_results[d]['deepsets'] = results
    
    # Evaluate GraphSAGE
    if graphsage_model:
        print(f"  Evaluating GraphSAGE d={d}...")
        graphs = shared_test_data[d]['graphs']
        results = evaluate_accuracy_metrics(
            graphsage_model,
            graphs,
            labels,
            NUM_ACCURACY_RUNS,
            is_graph_model=True
        )
        accuracy_results[d]['graphsage'] = results

print("\n✓ Accuracy evaluation complete")

## 9. Statistical Tests

In [ ]:
def wilcoxon_test(deepsets_accs: List[float], graphsage_accs: List[float]) -> Dict:
    """Perform Wilcoxon signed-rank test."""
    if len(deepsets_accs) != len(graphsage_accs) or len(deepsets_accs) < 2:
        return {'test': 'Wilcoxon signed-rank', 'statistic': 0, 'p_value': 1.0, 'significant': False}
    
    statistic, p_value = wilcoxon(deepsets_accs, graphsage_accs, alternative='two-sided')
    
    n = len(deepsets_accs)
    z = stats.norm.ppf(p_value / 2) if p_value > 0 else 0
    r = abs(z) / np.sqrt(n)
    
    return {
        'test': 'Wilcoxon signed-rank',
        'statistic': float(statistic),
        'p_value': float(p_value),
        'significant': p_value < ALPHA,
        'effect_size_r': float(r),
        'interpretation': 'large' if r > 0.5 else ('medium' if r > 0.3 else 'small')
    }


def cohens_d(group1: List[float], group2: List[float]) -> float:
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = np.mean(group1), np.mean(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return float((mean1 - mean2) / pooled_std)


# Perform statistical tests
print("Performing statistical tests...")
statistical_tests = {}

deepsets_all_accs = []
graphsage_all_accs = []

for d in DISTANCES:
    if d not in accuracy_results:
        continue
    
    if 'deepsets' in accuracy_results[d] and 'graphsage' in accuracy_results[d]:
        deepsets_accs = accuracy_results[d]['deepsets']['all_accuracies']
        graphsage_accs = accuracy_results[d]['graphsage']['all_accuracies']
        
        deepsets_all_accs.extend(deepsets_accs)
        graphsage_all_accs.extend(graphsage_accs)
        
        wilcoxon_result = wilcoxon_test(deepsets_accs, graphsage_accs)
        cohens_d_val = cohens_d(deepsets_accs, graphsage_accs)
        
        statistical_tests[d] = {
            'wilcoxon': wilcoxon_result,
            'cohens_d': float(cohens_d_val),
            'effect_size_interpretation': 'large' if abs(cohens_d_val) > 0.8 else ('medium' if abs(cohens_d_val) > 0.5 else 'small')
        }

# Overall Wilcoxon test
if len(deepsets_all_accs) > 0 and len(graphsage_all_accs) > 0:
    overall_wilcoxon = wilcoxon_test(deepsets_all_accs, graphsage_all_accs)
    overall_cohens_d = cohens_d(deepsets_all_accs, graphsage_all_accs)
    
    statistical_tests['overall'] = {
        'wilcoxon': overall_wilcoxon,
        'cohens_d': float(overall_cohens_d),
        'effect_size_interpretation': 'large' if abs(overall_cohens_d) > 0.8 else ('medium' if abs(overall_cohens_d) > 0.5 else 'small')
    }

print("✓ Statistical testing complete")
print(f"\nOverall Wilcoxon test: p={statistical_tests.get('overall', {}).get('wilcoxon', {}).get('p_value', 'N/A'):.4f}")

## 10. Visualization

In [ ]:
def plot_accuracy_vs_distance(accuracy_results, statistical_tests, output_path):
    """Graph 1: Accuracy vs Code Distance with error bars."""
    distances = sorted([d for d in accuracy_results.keys() if 'deepsets' in accuracy_results[d] and 'graphsage' in accuracy_results[d]])
    
    deepsets_accs = [accuracy_results[d]['deepsets']['accuracy_mean'] for d in distances]
    deepsets_cis = [accuracy_results[d]['deepsets']['accuracy_ci_95'] for d in distances]
    graphsage_accs = [accuracy_results[d]['graphsage']['accuracy_mean'] for d in distances]
    graphsage_cis = [accuracy_results[d]['graphsage']['accuracy_ci_95'] for d in distances]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(distances))
    width = 0.35
    
    ax.errorbar(x - width/2, deepsets_accs, yerr=deepsets_cis, 
               fmt='o-', label='DeepSets', color='steelblue', 
               capsize=5, capthick=2, linewidth=2, markersize=8)
    ax.errorbar(x + width/2, graphsage_accs, yerr=graphsage_cis,
               fmt='s-', label='GraphSAGE', color='coral',
               capsize=5, capthick=2, linewidth=2, markersize=8)
    
    ax.set_xlabel('Code Distance (d)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title('Accuracy vs Code Distance (95% CI)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'd={d}' for d in distances])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.1])
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

# Generate plots
print("Generating plots...\n")
plot_accuracy_vs_distance(
    accuracy_results, 
    statistical_tests, 
    PLOTS_DIR / "accuracy_vs_distance.png"
)

print("\n✓ All plots generated")

## 11. Save Results

In [ ]:
# Convert numpy types for JSON
def convert_to_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_json_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    else:
        return obj

# Save results
comparison_results = {
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'experiment_name': EXPERIMENT_NAME,
        'distances': DISTANCES,
        'test_samples_per_distance': TEST_SAMPLES_PER_DISTANCE,
        'num_benchmark_runs': NUM_BENCHMARK_RUNS,
        'num_accuracy_runs': NUM_ACCURACY_RUNS,
        'significance_level': ALPHA,
        'seed': SEED
    },
    'parameter_comparison': param_comparison,
    'model_info': {
        'deepsets': deepsets_info,
        'graphsage': graphsage_info
    }
}

with open(OUTPUT_RESULTS_DIR / "comparison_results.json", 'w') as f:
    json.dump(convert_to_json_serializable(comparison_results), f, indent=2)

# Save statistical tests
with open(OUTPUT_RESULTS_DIR / "statistical_tests.json", 'w') as f:
    json.dump(convert_to_json_serializable(statistical_tests), f, indent=2)

print("✓ All results saved to JSON files")
print(f"  Results: {OUTPUT_RESULTS_DIR}")

## 12. Summary Report

In [ ]:
print("=" * 80)
print("DEEPSETS vs GRAPHSAGE COMPARISON REPORT")
print("=" * 80)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Experiment: {EXPERIMENT_NAME}")
print()

print("MODEL PARAMETERS")
print("-" * 80)
print(f"DeepSets: {deepsets_info.get('num_parameters', 0):,} parameters")
print(f"GraphSAGE: {graphsage_info.get('num_parameters', 0):,} parameters")
print()

print("ACCURACY BY DISTANCE")
print("-" * 80)
for d in DISTANCES:
    if d in accuracy_results:
        ds_acc = accuracy_results[d].get('deepsets', {}).get('accuracy_mean', 0)
        gs_acc = accuracy_results[d].get('graphsage', {}).get('accuracy_mean', 0)
        diff = ds_acc - gs_acc
        winner = "DeepSets" if diff > 0 else "GraphSAGE"
        print(f"d={d}: DeepSets={ds_acc:.4f}, GraphSAGE={gs_acc:.4f}, Diff={diff:+.4f} ({winner})")

print()
print("STATISTICAL SIGNIFICANCE")
print("-" * 80)
if 'overall' in statistical_tests:
    overall = statistical_tests['overall']
    print(f"Overall Wilcoxon p-value: {overall['wilcoxon']['p_value']:.6f}")
    print(f"Significant at α={ALPHA}: {'YES' if overall['wilcoxon']['significant'] else 'NO'}")
    print(f"Cohen's d: {overall['cohens_d']:.4f} ({overall['effect_size_interpretation']})")

print()
print("=" * 80)